# Tenet

Denis Barilov, January 2026

Tenet is a model to compress images into embeggings. To see the process of its creation you can dive into the full project in a file `ImageCompression.ipynb` (this code is a fragment from the main part).

In [ ]:
!pip install optuna torchmetrics

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data import Dataset
from PIL import Image
import torch.nn as nn
from torch.utils.data import random_split
import json
import os
from tqdm.notebook import tqdm
from torchvision.models.resnet import BasicBlock
import matplotlib.ticker as ticker
import numpy as np
from torchmetrics.image import StructuralSimilarityIndexMeasure, MultiScaleStructuralSimilarityIndexMeasure

In [ ]:
data_path = "/content/data"  # REPLACE WITH YOUR IMAGES DIRECTORY
data_prefix = "/content"  # REPLACE WITH A DIRECTORY IN WHICH YOU WANT TO SAVE MODELS

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.image_files = os.listdir(folder_path)
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder_path, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image


transform = transforms.Compose([
    transforms.ToTensor(),
])

dataset = CustomImageDataset(data_path, transform=transform)

In [ ]:
def torch_train_test_split(dataset, shares = [.8, .1, .1], **params):
    if sum(shares) == 1:
        total_size = len(dataset)
        lengths = [int(share * total_size) for share in shares]
        lengths[-1] = total_size - sum(lengths[:-1])
        return random_split(dataset, lengths, **params)
    else:
        return random_split(dataset, shares, **params)

RANDOM_SEED = 42
generator = torch.Generator().manual_seed(RANDOM_SEED)

train_dataset, val_dataset, test_dataset = torch_train_test_split(dataset, generator=generator)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
experiments = dict()

In [ ]:
def load_model(experiments, model, model_name, model_path = None, force = False):
    if model_name in experiments.keys() and not force:
        return

    if model_path is None:
        model_path = f'/content/{model_name}.pth'

    model.load_state_dict(torch.load(model_path, map_location=device))
    experiments[model_name] = {
        "model": model,
        "model_name": model_name,
    }

def save_experiment(experiment):
    model = experiment['model']
    experiment['model'] = type(model).__name__
    with open(f"{data_prefix}/{experiment['model_name']}.json", "w", encoding="utf-8") as f:
        json.dump(experiment, f)
    experiment['model'] = model

    torch.save(model.state_dict(), f"{data_prefix}/{experiment['model_name']}.pth")

In [ ]:
def need_stop(records, delta, size, threshold):
    if len(records) < size + 1:
        return False
    while len(records) > size + 1:
        records.pop(0)
    count = sum([records[i] - records[i+1] < 1e-6 for i in range(len(records)-1)])
    return count >= threshold * size

In [ ]:
def train(model, optimizer, criterion = nn.MSELoss(), max_epochs = 150, early_stop_params=[1e-4, 8, 0.75], noize_intensity=0, **params):
    max_epochs = 150
    model.train()

    history = []
    experiment = {
        "history": history,
        "max_epochs": max_epochs,
        "model": model,
    }
    experiment.update(params)

    loss_cycle = []
    for epoch in tqdm(range(max_epochs)):
        model.train()
        train_loss = 0.0
        for images in train_loader:
            images = images.to(device)

            if noize_intensity != 0:
                noisy_images = images + torch.randn_like(images) * noize_intensity
                noisy_images = torch.clamp(noisy_images, 0.0, 1.0)
                outputs = model(noisy_images)
            else:
                outputs = model(images)

            loss = criterion(outputs, images)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images in val_loader:
                images = images.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, images).item()

        history.append({
            "epoch": epoch,
            "epoch_train_loss": train_loss / len(train_loader),
            "epoch_val_loss": val_loss / len(val_loader),
        })

        if early_stop_params:
            loss_cycle.append(val_loss / len(val_loader))
            if need_stop(loss_cycle, *early_stop_params):
                break

    return experiment

In [ ]:
class TonePool2d(nn.Module):
    def __init__(self, target=(0.5, 0.5, 0.5)):
        super().__init__()
        self.register_buffer("target", torch.tensor(target).view(1, 3, 1, 1, 1))

    def forward(self, x):
        B, C, H, W = x.shape  # C=3

        x_flattened = x.unfold(2, 2, 2).unfold(3, 2, 2).reshape(B, 3, H//2, W//2, 4)  # -> B x 3 x H/2 x W/2 x 4
        dist = torch.abs(x_flattened - self.target).sum(dim=1)  # -> B x H/2 x W/2 x 4

        idx = dist.argmin(dim=3)  # -> B x H/2 x W/2
        idx_expanded = idx.unsqueeze(1).unsqueeze(4).expand(-1, 3, -1, -1, -1)  # -> B x 3 x H/2 x W/2 x 1

        pixels = torch.gather(x_flattened, 4, idx_expanded)  # -> B x 3 x H/2 x W/2 x 1
        return pixels.squeeze(-1)  # -> B x 3 x H/2 x W/2


In [ ]:
class Tenet(nn.Module):
    def __init__(self, embedded_channels = 128, tone_layers = 3):
        super(Tenet, self).__init__()

        if tone_layers <= 2:
            self.skip_encoder = nn.Sequential(
                *[nn.AvgPool2d(kernel_size=2, stride=2) for _ in range(tone_layers-2)]
            )  # -> 3 x 400^{-tone_layers} x 320^{-tone_layers}
        else:
            self.skip_encoder = nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=2),
                nn.AvgPool2d(kernel_size=2, stride=2),
                *[TonePool2d() for _ in range(tone_layers-2)]
            )  # -> 3 x 400^{-tone_layers} x 320^{-tone_layers}


        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, padding_mode='reflect'),  # -> 16 x 200 x 160
            nn.BatchNorm2d(16),
            nn.ReLU(),

            BasicBlock(inplanes=16, planes=16),

            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, padding_mode='reflect'),  # -> 32 x 100 x 80
            nn.BatchNorm2d(32),
            nn.ReLU(),

            BasicBlock(inplanes=32, planes=32),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, padding_mode='reflect'),  # -> 64 x 50 x 40
            nn.BatchNorm2d(64),
            nn.ReLU(),

            BasicBlock(inplanes=64, planes=64),

            nn.Conv2d(64, embedded_channels, kernel_size=3, stride=2, padding=1, padding_mode='reflect'),  # -> embedded_channels x 25 x 20
            nn.BatchNorm2d(embedded_channels),
            nn.ReLU(),

            BasicBlock(inplanes=embedded_channels, planes=embedded_channels),
        )

        self.skip_decoder = nn.Sequential(
            nn.Upsample(scale_factor=2**tone_layers, mode='bicubic'),  # -> 3 x 400 x 320
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedded_channels, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> 64 x 50 x 40
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> 32 x 100 x 80
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> 16 x 200 x 160
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(16, 8, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> 8 x 400 x 320
            nn.Sigmoid()
        )

        self.combiner = nn.Sequential(  # (8+3) x 400 x 320
            nn.Conv2d(11, 3, kernel_size=5, stride=1, padding=2, padding_mode='reflect'),  # -> 3 x 400 x 320
            nn.Sigmoid()
        )

    def forward(self, x):
        skipped = self.skip_encoder(x)
        encoded = self.encoder(x)

        upsampled = self.skip_decoder(skipped)
        decoded = self.decoder(encoded)
        combined = torch.cat([decoded, upsampled], dim=1)

        return self.combiner(combined)

    def compress(self, x):
        skipped = self.skip_encoder(x)
        encoded = self.encoder(x)
        return (skipped, encoded)

    def decompress(self, embedding):
        skipped, encoded = embedding

        upsampled = self.skip_decoder(skipped)
        decoded = self.decoder(encoded)
        combined = torch.cat([decoded, upsampled], dim=1)

        return self.combiner(combined)


In [ ]:
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
ms_ssim = MultiScaleStructuralSimilarityIndexMeasure(data_range=1.0).to(device)
ms_sim_loss = lambda outputs, images: 1 - ms_ssim(outputs, images)
mse_loss = nn.MSELoss()

def get_combined_loss(alpha):
    return lambda outputs, images: mse_loss(outputs, images) + alpha * ms_sim_loss(outputs, images)

In [ ]:
lr = 0.0027703296207831733
weight_decay = 4.253026198661108e-06

In [ ]:
model = Tenet().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = get_combined_loss(0.5)

experiment = train(model, optimizer, criterion, model_name="Tenet-128-3_MSSSIM_05", early_stop_params=[1e-6, 20, 0.9], noize_intensity=0, hyperparameters={"lr": lr, "weight_decay": weight_decay, "loss": "combined", "alpha": 0.5}, model_hyperparameters={"embedded_channels": 128, "tone_layers": 3})

experiments[experiment['model_name']] = experiment
save_experiment(experiment)

In [ ]:
model = Tenet().to(device)
load_model(experiments, model, 'Tenet-128-3_MSSSIM_05')